# 🌦️ Conceptual Background: Scaling vs Semantic Consistency in VRPTW

**Author:** Miguel Vásquez  
**Date:** October 2025  
**Role:** Data Engineer | Machine Learning Engineer  

---

## 1. Introduction
In this notebook, we extend the spatial–temporal framework developed in `03_distance_matrix_and_constraints.ipynb` by introducing **stochastic variability** and **robustness evaluation**.

Real-world logistics systems operate under uncertainty — travel times and distances fluctuate due to factors such as congestion, weather, or road incidents.
Our objective here is to **simulate these uncertainties** through controlled perturbations and evaluate how they affect **route feasibility**, **operational cost**, and **solution stability**.

This stage bridges deterministic optimization and realistic simulation — a key step toward our final **visual scenario-based evaluation**.

---

## 2. Core Components
In this notebook we will:
- Load the processed **distance** and **time matrices** from `data/processed/`.
- Use the new utilities in `scenario_utils.py` to:
    - Generate stochastic perturbations (`generate_scenarios`).
    - Simulate localized disruptions (`simulate_accident`).
    - Aggregate variability through percentile-based robustness metrics (`percentile_matrix`).
- Visualize scenario dispersion and identify potential instability zones in the network.

---

## 3. Objectives
By the end of this notebook, we will have:
- A robust dataset of **time and distance scenarios** representing uncertainty in transportation conditions.
- Quantitative insight into **how variability propagates** across the network.
- Preliminary visualizations to assess **robustness and sensitivity** before solver integration.

---

## 4. Expected Output
- Scenario matrices stored in `data/scenarios/`, e.g.:
    - `time_scenario_001.csv`, `time_scenario_002.csv`, …
    - `distance_scenario_001.csv`, etc.
- Aggregated robustness matrices (e.g., 90th percentile time matrix).
- Visual analysis of scenario dispersion and local disruptions.

---

## 5. Setup & Data Loading
In this section, we initialize dependencies, define working directories, and load the processed matrices generated in the previous notebook.
This will serve as the foundation for subsequent scenario generation steps.

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

# Add the parent directory to sys.path so 'src' can be imported
sys.path.append(str(Path.cwd().parent))

from src.data_utils import ensure_project_root, load_processed_data
from src.scenario_utils import generate_scenarios, simulate_accident, percentile_matrix

# Ensure we're in the project root
ensure_project_root()

# Configure display and random seed
pd.set_option("display.max_columns", None)
np.random.seed(42)

#paths
distance_path = "data/processed/vrptw_distance_matrix.csv"
time_path = "data/processed/vrptw_time_matrix.csv"

# Load processed data
distance_df  = load_processed_data(distance_path)
time_df  = load_processed_data(time_path)

display(distance_df.head(3))
display(time_df.head(3))

Changed working directory to project root: `c:\Users\Miguel\portfolio\Route-Optimization-VRPTW`

Loading processed dataset from: data/processed/vrptw_distance_matrix.csv
✅ Loaded dataset with 5656 rows and 5656 columns.
Loading processed dataset from: data/processed/vrptw_time_matrix.csv
✅ Loaded dataset with 5656 rows and 5656 columns.


C101_1    C101_2    C101_3    C101_4    C101_5    C101_6    C101_7  \
C101_1  0.000000  0.225734  0.249516  0.196254  0.220519  0.184134  0.231707   
C101_2  0.225734  0.000000  0.024390  0.039901  0.031579  0.048329  0.054026   
C101_3  0.249516  0.024390  0.000000  0.058110  0.039901  0.068668  0.054026   

          C101_8    C101_9   C101_10   C101_11   C101_12   C101_13   C101_14  \
C101_1  0.195122  0.220519  0.244809  0.202096  0.237610  0.455098  0.358958   
C101_2  0.058008  0.073684  0.077616  0.108052  0.105967  0.295469  0.256714   
C101_3  0.071761  0.077616  0.073684  0.116017  0.105967  0.278897  0.249666   

         C101_15   C101_16   C101_17   C101_18   C101_19   C101_20   C101_21  \
C101_1  0.466994  0.422102  0.475925  0.382857  0.402744  0.450667  0.105263   
C101_2  0.318740  0.301111  0.335011  0.296754  0.327124  0.348050  0.270400   
C101_3  0.303442  0.290042  0.320491  0.290678  0.321622  0.338519  0.290550   

         C101_22   C101_23   C101_24   C101_25   C101_26   C101_27   C101_28  \
C101_1  0.108052  0.128649  0.140263  0.157895  0.159767  0.169259  0.180602   
C101_2  0.251005  0.264754  0.239073  0.304150  0.287043  0.263544  0.302822   
C101_3  0.270400  0.283210  0.255899  0.322195  0.304150  0.278897  0.319084   

         C101_29   C101_30   C101_31   C101_32   C101_33   C101_34   C101_35  \
C101_1  0.189051  0.210526  0.219179  0.364945  0.338519  0.358238  0.342317   
C101_2  0.280647  0.342692  0.307223  0.545611  0.502326  0.517964  0.479962   
C101_3  0.295112  0.358804  0.320491  0.563842  0.519214  0.534358  0.494611   

         C101_36   C101_37   C101_38   C101_39   C101_40   C101_41   C101_42  \
C101_1  0.411335  0.373433  0.418177  0.438358  0.425445  0.249516  0.225734   
C101_2  0.582445  0.505924  0.566986  0.583930  0.550500  0.475219  0.451467   
C101_3  0.599557  0.519842  0.582000  0.598520  0.563318  0.499033  0.475219   

         C101_43   C101_44   C101_45   C101_46   C101_47   C101_48   C101_49  \
C101_1  0.231549  0.197210  0.258031  0.265648  0.243446  0.211051  0.274671   
C101_2  0.456835  0.421797  0.483197  0.489575  0.466555  0.432305  0.496765   
C101_3  0.480322  0.445128  0.506635  0.512722  0.489575  0.455098  0.519592   

         C101_50   C101_51   C101_52   C101_53   C101_54   C101_55   C101_56  \
C101_1  0.222301  0.264392  0.290550  0.241646  0.550393  0.488259  0.427348   
C101_2  0.440431  0.482434  0.508994  0.454179  0.768365  0.708022  0.647112   
C101_3  0.462823  0.504731  0.531296  0.475925  0.792753  0.732388  0.671475   

         C101_57   C101_58   C101_59   C101_60   C101_61   C101_62   C101_63  \
C101_1  0.548780  0.426829  0.549184  0.427348  0.551299  0.265648  0.211051   
C101_2  0.770093  0.648481  0.771818  0.650528  0.775470  0.466394  0.405866   
C101_3  0.794428  0.672794  0.796100  0.674767  0.799642  0.490636  0.430062   

         C101_64   C101_65   C101_66   C101_67   C101_68   C101_69   C101_70  \
C101_1  0.161098  0.258031  0.148201  0.197210  0.142483  0.249516  0.190348   
C101_2  0.345496  0.464489  0.342921  0.402989  0.342112  0.463415  0.402439   
C101_3  0.369620  0.488826  0.367214  0.427348  0.366459  0.487805  0.426829   

         C101_71   C101_72   C101_73   C101_74   C101_75   C101_76   C101_77  \
C101_1  0.628226  0.607159  0.279668  0.599250  0.228447  0.190348  0.557199   
C101_2  0.701257  0.662545  0.471004  0.677877  0.411155  0.036585  0.621558   
C101_3  0.717608  0.677637  0.495020  0.694779  0.435057  0.060976  0.637621   

         C101_78   C101_79   C101_80   C101_81   C101_82   C101_83   C101_84  \
C101_1  0.561052  0.537358  0.551591  0.563318  0.507778  0.373433  0.342317   
C101_2  0.647787  0.605667  0.640477  0.672511  0.582445  0.353351  0.325437   
C101_3  0.665454  0.622140  0.658339  0.691697  0.599557  0.364945  0.337991   

         C101_85   C101_86   C101_87   C101_88   C101_89   C101_90   C101_91  \
C101_1  0.330516  0.318970  0.280394  0.270130  0.290042  0.261023  0.219179  

C101_1    C101_2    C101_3    C101_4    C101_5    C101_6    C101_7  \
C101_1  0.000000  0.225734  0.249516  0.196254  0.220519  0.184134  0.231707   
C101_2  0.225734  0.000000  0.024390  0.039901  0.031579  0.048329  0.054026   
C101_3  0.249516  0.024390  0.000000  0.058110  0.039901  0.068668  0.054026   

          C101_8    C101_9   C101_10   C101_11   C101_12   C101_13   C101_14  \
C101_1  0.195122  0.220519  0.244809  0.202096  0.237610  0.455098  0.358958   
C101_2  0.058008  0.073684  0.077616  0.108052  0.105967  0.295469  0.256714   
C101_3  0.071761  0.077616  0.073684  0.116017  0.105967  0.278897  0.249666   

         C101_15   C101_16   C101_17   C101_18   C101_19   C101_20   C101_21  \
C101_1  0.466994  0.422102  0.475925  0.382857  0.402744  0.450667  0.105263   
C101_2  0.318740  0.301111  0.335011  0.296754  0.327124  0.348050  0.270400   
C101_3  0.303442  0.290042  0.320491  0.290678  0.321622  0.338519  0.290550   

         C101_22   C101_23   C101_24   C101_25   C101_26   C101_27   C101_28  \
C101_1  0.108052  0.128649  0.140263  0.157895  0.159767  0.169259  0.180602   
C101_2  0.251005  0.264754  0.239073  0.304150  0.287043  0.263544  0.302822   
C101_3  0.270400  0.283210  0.255899  0.322195  0.304150  0.278897  0.319084   

         C101_29   C101_30   C101_31   C101_32   C101_33   C101_34   C101_35  \
C101_1  0.189051  0.210526  0.219179  0.364945  0.338519  0.358238  0.342317   
C101_2  0.280647  0.342692  0.307223  0.545611  0.502326  0.517964  0.479962   
C101_3  0.295112  0.358804  0.320491  0.563842  0.519214  0.534358  0.494611   

         C101_36   C101_37   C101_38   C101_39   C101_40   C101_41   C101_42  \
C101_1  0.411335  0.373433  0.418177  0.438358  0.425445  0.249516  0.225734   
C101_2  0.582445  0.505924  0.566986  0.583930  0.550500  0.475219  0.451467   
C101_3  0.599557  0.519842  0.582000  0.598520  0.563318  0.499033  0.475219   

         C101_43   C101_44   C101_45   C101_46   C101_47   C101_48   C101_49  \
C101_1  0.231549  0.197210  0.258031  0.265648  0.243446  0.211051  0.274671   
C101_2  0.456835  0.421797  0.483197  0.489575  0.466555  0.432305  0.496765   
C101_3  0.480322  0.445128  0.506635  0.512722  0.489575  0.455098  0.519592   

         C101_50   C101_51   C101_52   C101_53   C101_54   C101_55   C101_56  \
C101_1  0.222301  0.264392  0.290550  0.241646  0.550393  0.488259  0.427348   
C101_2  0.440431  0.482434  0.508994  0.454179  0.768365  0.708022  0.647112   
C101_3  0.462823  0.504731  0.531296  0.475925  0.792753  0.732388  0.671475   

         C101_57   C101_58   C101_59   C101_60   C101_61   C101_62   C101_63  \
C101_1  0.548780  0.426829  0.549184  0.427348  0.551299  0.265648  0.211051   
C101_2  0.770093  0.648481  0.771818  0.650528  0.775470  0.466394  0.405866   
C101_3  0.794428  0.672794  0.796100  0.674767  0.799642  0.490636  0.430062   

         C101_64   C101_65   C101_66   C101_67   C101_68   C101_69   C101_70  \
C101_1  0.161098  0.258031  0.148201  0.197210  0.142483  0.249516  0.190348   
C101_2  0.345496  0.464489  0.342921  0.402989  0.342112  0.463415  0.402439   
C101_3  0.369620  0.488826  0.367214  0.427348  0.366459  0.487805  0.426829   

         C101_71   C101_72   C101_73   C101_74   C101_75   C101_76   C101_77  \
C101_1  0.628226  0.607159  0.279668  0.599250  0.228447  0.190348  0.557199   
C101_2  0.701257  0.662545  0.471004  0.677877  0.411155  0.036585  0.621558   
C101_3  0.717608  0.677637  0.495020  0.694779  0.435057  0.060976  0.637621   

         C101_78   C101_79   C101_80   C101_81   C101_82   C101_83   C101_84  \
C101_1  0.561052  0.537358  0.551591  0.563318  0.507778  0.373433  0.342317   
C101_2  0.647787  0.605667  0.640477  0.672511  0.582445  0.353351  0.325437   
C101_3  0.665454  0.622140  0.658339  0.691697  0.599557  0.364945  0.337991   

         C101_85   C101_86   C101_87   C101_88   C101_89   C101_90   C101_91  \
C101_1  0.330516  0.318970  0.280394  0.270130  0.290042  0.261023  0.219179  

## 6. Scenario Generation & Visualization

Having established the deterministic distance and time matrices, 
we now introduce *scenario-based variability* to emulate real-world logistics dynamics.

This section focuses on:
- **Generating stochastic scenarios** that reflect travel-time uncertainty (e.g., congestion, weather, or accidents).
- **Visualizing scenario diversity** through statistical summaries and comparisons.
- **Preparing scenario datasets** that will serve as stress tests for the solver.

We’ll use three key functions from `src/scenario_utils.py`:
1. `generate_scenarios()` — create multiple perturbed matrices via lognormal noise.
2. `simulate_accident()` — inject local disruptions (e.g., blocked roads).
3. `percentile_matrix()` — compute aggregated (e.g., 90th percentile) matrices to summarize robustness.

Each generated scenario will represent an alternative world where travel times slightly differ from the baseline,
allowing the optimization engine to identify resilient routing strategies.

In [2]:
# Generate stochastic time scenarios
scenarios = generate_scenarios(
    distance_df,
    n_scenarios=50,
    sigma=0.15,
    edge_level=False,
    seed=42
)

display(Markdown(f"### Generated {len(scenarios)} stochastic time scenarios"))
display(Markdown(f"Each scenario matrix has shape: {scenarios[0].shape}"))

### Generated 50 stochastic time scenarios

Each scenario matrix has shape: (5656, 5656)

In [3]:
# Compare baseline vs first 3 scenarios (sample of origin-destination pairs)
sample_pairs = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3)]  # fila, columna
rows = []

for i, j in sample_pairs:
    row = {
        "Origin": distance_df.index[i],
        "Destination": distance_df.columns[j],
        "Baseline": distance_df.iloc[i, j]
    }
    for k in range(3):  # Scenarios 1-3
        row[f"Scenario_{k+1}"] = scenarios[k].iloc[i, j]
    rows.append(row)

summary = pd.DataFrame(rows)
display(Markdown("#### Baseline vs first 3 scenarios (selected OD pairs)"))
display(summary)


#### Baseline vs first 3 scenarios (selected OD pairs)

,Origin,Destination,Baseline,Scenario_1,Scenario_2,Scenario_3
0,C101_1,C101_2,0.225734,0.236291,0.193129,0.252629
1,C101_1,C101_3,0.249516,0.261186,0.213477,0.279246
2,C101_2,C101_3,0.024390,0.025531,0.020867,0.027296
3,C101_2,C101_4,0.039901,0.041767,0.034138,0.044655
4,C101_3,C101_4,0.058110,0.060828,0.049717,0.065034


### 6.1 Localized Accident Simulation
#### Concept
In real-world routing operations, travel times are rarely static.
Unexpected events — such as **accidents, congestion, or road closures** — can create **local disruptions** that significantly alter route feasibility and total cost.

In this section, we simulate such an event by **amplifying travel time** between two specific nodes.
This will allow us to:
- Observe the propagation of local delays,
- Assess the robustness of our routing model under disruptions.

We’ll use our helper function `simulate_accident()` from `src/scenario_utils.py` to multiply the travel time between two nodes by a delay factor (e.g., 5×).

In [4]:
node_i = "C101_10"
node_j = "C101_25"

disrupted_matrix = simulate_accident(time_df, node_i, node_j, factor=5.0)

comparison = pd.DataFrame({
    "Original_Time": [time_df.loc[node_i, node_j]],
    "Disrupted_Time": [disrupted_matrix.loc[node_i, node_j]]
}, index=[f"{node_i} → {node_j}"])

display(Markdown("### 🧩 Accident Simulation Result"))
display(comparison)


### 🧩 Accident Simulation Result

,Original_Time,Disrupted_Time
C101_10 → C101_25,0.279668,1.39834


### 6.2 Scenario Variability & Robustness Analysis
Even with stochastic variations in travel times, operational planners often prefer to **plan for robust scenarios** — ones that hedge against uncertainty by considering slightly pessimistic (but realistic) conditions.

A common approach is to aggregate multiple stochastic realizations (scenarios) into a **percentile-based travel time matrix**.  
For example, the **90th percentile** (P90) matrix represents a conservative estimate — values below which 90% of simulated travel times fall.

This provides a more **risk-aware** foundation for optimization:  
- P50 (median) = "most likely" scenario  
- P90 = "cautious but realistic" scenario  
- P100 = "worst case" (not typically used in practice)

In [5]:
# Compute percentile-based robustness matrix (e.g., 90th percentile)
percentile_90_matrix = percentile_matrix(scenarios, q=90)

# Display a small comparison with baseline
comparison_robust = pd.DataFrame({
    "Baseline": time_df.values.flatten()[:5],
    "P90_Robust": percentile_90_matrix.values.flatten()[:5]
})

display(Markdown("### Percentile-based Robustness (P90 Sample)"))
display(comparison_robust)

display(Markdown(f"✅ Generated percentile (P90) matrix with shape: {percentile_90_matrix.shape}"))

### Percentile-based Robustness (P90 Sample)

,Baseline,P90_Robust
0,0.000000,0.000000
1,0.225734,0.257801
2,0.249516,0.284962
3,0.196254,0.224134
4,0.220519,0.251846


✅ Generated percentile (P90) matrix with shape: (5656, 5656)

### 6.3 Export Scenario Matrices for Optimization
At this stage, we have prepared two scenario-based variants of our baseline travel-time matrix:
| Matrix | Description |
|:--|:--|
| **P90 Robust Matrix** | Aggregates stochastic variations to represent a conservative travel-time estimate. |
| **Accident Matrix** | Reflects a localized disruption due to an accident between two nodes. |

Exporting these ensures **reproducibility** and smooth continuation to the next notebook:  
`05 - Optimization & Routing`.

In [6]:
from src.data_utils import save_processed_data

# Save matrices to data/processed/
robust_path = save_processed_data(percentile_90_matrix, "vrptw_time_matrix_robust_p90.csv")
accident_path = save_processed_data(disrupted_matrix, "vrptw_time_matrix_accident.csv")

✅ Data saved successfully at: data/processed\vrptw_time_matrix_robust_p90.csv
Rows: 5656 | Columns: 5656
✅ Data saved successfully at: data/processed\vrptw_time_matrix_accident.csv
Rows: 5656 | Columns: 5656


## 7. Summary & Next Steps

In this notebook, we extended our VRPTW modeling pipeline beyond static assumptions by introducing **scenario-based variability** and **robustness analysis**.

### Key Achievements
- Generated **stochastic travel-time scenarios** using controlled lognormal perturbations.
- Simulated **localized disruptions** (e.g., accidents or blockages) to test model resilience.
- Aggregated variability through a **percentile-based robustness matrix (P90)**, capturing 90% of the expected travel-time distribution.
- Exported representative scenario matrices (`_robust_p90.csv`, `_accident.csv`) for downstream optimization.

### Next Steps — Optimization & Routing
In the next notebook, `05_optimization_and_routing.ipynb`, we will:
- Integrate these scenario matrices into the **solver’s optimization engine**.
- Apply **VRPTW constraints** (capacity, time windows, service durations).
- Compare baseline vs. robust vs. disrupted route plans.
- Analyze **solution cost stability** and **resilience** under uncertainty.

> With this foundation, we now transition from **data simulation** to **decision optimization** — where the solver will learn to navigate uncertainty, not just observe it.